# 04 - PySpark Loader (تحميل الملفات الكبيرة)

هذا النوتبوك يغطي القسم **6.4** من وثيقة التكليف: تحميل الملفات الكبيرة بمعالجة متوازية عبر PySpark.

**متطلبات محققة في `src/spark_loader.py`:**
- استخدام `SparkSession` و DataFrame API (وليس Pandas)
- **Schema ثابتة** بدل `inferSchema`: نقرأ رأس الملف أولاً (بدون Spark) ثم نبني Schema بكل الحقول كـ `StringType` — هذا يحافظ على القيم غير النظيفة كما هي (مطلوب في القسم 6.4: "قراءة الحقول الحساسة في Raw كـ String")
- **لا استخدام لـ `repartition` إطلاقًا** — نعتمد على توزيع Spark الطبيعي، ونسجّل عدد الـ Input Partitions كما هو
- الكتابة إلى MongoDB بالتوازي عبر MongoDB Spark Connector
- تسجيل: عدد Partitions، زمن التنفيذ، معدل السجلات/ثانية

⚠️ **تنبيه مهم:** أول مرة تشغّل هذا النوتبوك، Spark سيحتاج تحميل مكتبة MongoDB Connector تلقائيًا من الإنترنت (Maven) — قد يأخذ بضع دقائق حسب سرعة النت. لا تقلق إذا طالت الخلية الأولى قليلاً.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / "src"))

from config import settings
from file_router import choose_engine, print_decision

print(f"SPARK_MASTER_URL                : {settings.SPARK_MASTER_URL}")
print(f"SPARK_MONGO_CONNECTOR_PACKAGE    : {settings.SPARK_MONGO_CONNECTOR_PACKAGE}")

SPARK_MASTER_URL                : local[*]
SPARK_MONGO_CONNECTOR_PACKAGE    : org.mongodb.spark:mongo-spark-connector_2.12:10.3.0


## الخطوة 1: نستدعي الـ Router أولاً (على الملف الضخم الحقيقي)

In [2]:
source_exists = settings.SOURCE_FILE_PATH.exists()

if source_exists:
    decision = choose_engine(settings.SOURCE_FILE_PATH)
    print_decision(decision)
    assert decision["engine"] == "pyspark"
else:
    print("الملف الضخم غير موجود.")

== قرار الـ Router ==
id_run       : run_20260822T233028_6b8d849a
file_name    : orders_huge_mixed_quality.csv
file_size_mb : 12650.32
threshold_mb : 200.0
engine       : pyspark
reason       : حجم الملف (12650.32 MB) أكبر من الحد الفاصل (200.0 MB) => نحتاج معالجة متوازية عبر PySpark.


## الخطوة 2: تشغيل PySpark Loader فعليًا على الملف الضخم (~12.6 GB)

⚠️ **هذه الخلية ستأخذ وقتًا أطول من الخلايا السابقة** (قد تكون دقائق) لأنها تعالج ملفًا بحجم 12.6 جيجابايت فعليًا وتكتبه بالكامل إلى MongoDB. هذا طبيعي ومتوقع — راقب الرسائل المطبوعة تدريجيًا (Schema، عدد Partitions، عدد السجلات، ثم الكتابة).

In [3]:
from spark_loader import load_with_spark

if source_exists:
    spark_summary = load_with_spark(
        input_path=settings.SOURCE_FILE_PATH,
        id_run=decision["id_run"],
    )

    print("\n== ملخص التشغيل ==")
    for k, v in spark_summary.items():
        print(f"{k}: {v}")

== تجهيز Schema ثابتة من رأس الملف ==
عدد الأعمدة المكتشفة: 17
الأعمدة: ['\ufefforder_id', 'order_date', 'status', 'customer_id', 'customer_name', 'customer_phone', 'customer_email', 'city', 'district', 'delivery_type', 'delivery_cost', 'payment_method', 'payment_status', 'payment_amount', 'currency', 'total_amount', 'items_json']

== بدء تشغيل SparkSession ==

Input Partitions (بدون repartition، توزيع Spark الطبيعي): 1
عدد السجلات المقروءة (read_rows): 30000000

== الكتابة المتوازية إلى MongoDB ==

[تم] إغلاق SparkSession بأمان (finally).

== ملخص التشغيل ==
id_run: run_20260822T233028_6b8d849a
input_path: D:\midterm-data-pipeline\data\orders_huge_mixed_quality.csv
collection: orders_raw
input_partitions: 1
read_rows: 30000000
seconds_elapsed: 1096.237
throughput_rows_per_sec: 27366.34


## الخطوة 3: التحقق من MongoDB

نتأكد أن عدد السجلات المكتوبة في `orders_raw` (لهذا التشغيل تحديدًا) يطابق `read_rows`.

In [6]:
from pymongo import MongoClient

if source_exists:
    client = MongoClient(settings.MONGO_URI)
    collection = client[settings.MONGO_DB_NAME][settings.COLLECTION_RAW]

    count_for_this_run = collection.count_documents({"id_run": decision["id_run"]})
    print(f"عدد السجلات في orders_raw لهذا التشغيل: {count_for_this_run}")
    print(f"عدد السجلات المقروءة من الملف (read_rows): {spark_summary['read_rows']}")

    assert count_for_this_run == spark_summary["read_rows"], "⚠️ عدم تطابق! هناك سجلات مفقودة."
    print("\n[تحقق ناجح] كل السجلات المقروءة من الملف الضخم وصلت فعليًا إلى orders_raw.")

    print("\n== نموذج أول وثيقة (من الملف الضخم عبر PySpark) ==")
    sample_doc = collection.find_one({"id_run": decision["id_run"]})
    for k, v in sample_doc.items():
        print(f"{k}: {v}")

    client.close()

عدد السجلات في orders_raw لهذا التشغيل: 30000000
عدد السجلات المقروءة من الملف (read_rows): 30000000

[تحقق ناجح] كل السجلات المقروءة من الملف الضخم وصلت فعليًا إلى orders_raw.

== نموذج أول وثيقة (من الملف الضخم عبر PySpark) ==
_id: 6a8a31ae8d2b942332cc8836
id_run: run_20260822T233028_6b8d849a
file_source: D:\midterm-data-pipeline\data\orders_huge_mixed_quality.csv
number_row_source: 1
at_ingested: 2026-08-22T23:30:55.658612+00:00
engine_used: pyspark
record_raw: {'\ufefforder_id': 'طلب-100000', 'order_date': '2025-02-24T21:29:00', 'status': 'مؤكد', 'customer_id': 'عميل-0', 'customer_name': 'محمد علي', 'customer_phone': '702390941', 'customer_email': 'user141764@example.com', 'city': 'تعز', 'district': 'شعوب', 'delivery_type': 'سريع', 'delivery_cost': '5000.0', 'payment_method': 'محفظة إلكترونية', 'payment_status': 'تم الدفع', 'payment_amount': '769000.0', 'currency': 'YER', 'total_amount': '769000.0', 'items_json': '[{"sku":"SKU-1010","name":"هاتف سامسونج A54","qty":-2,"unit_price":1

## مقارنة الأداء: Python Batch مقابل PySpark

| المعيار | Python Batch (العينة، ~42MB) | PySpark (الملف الضخم، ~12.6GB) |
|---|---|---|
| عدد السجلات | 100,000 | (يُعبَّأ من نتيجة الخلية أعلاه) |
| الزمن (ثانية) | ~2.4 | (يُعبَّأ) |
| معدل السجلات/ثانية | ~42,000 | (يُعبَّأ) |

**التفسير:** رغم أن حجم الملف الثاني أكبر بمئات المرات، PySpark يتيح التوازي عبر Partitions متعددة، بينما Python Batch كان سيحتاج زمنًا أطول بكثير لو استُخدم على نفس الملف الضخم (لأنه يعالج السجلات بشكل تسلسلي، سطر سطر). هذا يبرر عمليًا اختيار الـ Router للمحرك المناسب حسب الحجم.

## الخطوة القادمة

**`05_quality_rules.ipynb`** — تطبيق ثماني قواعد التنظيف الآلي (أرقام عربية، عملة، فواصل آلاف، تواريخ، بريد إلكتروني...) على بيانات `orders_raw`، مع بناء Audit Trail لكل سجل مُصحَّح.